# Animation: landing on the moving vehicle

Reads `platform_flight.csv`, written by `EINT_PLATFORM_eval.m` (flight on the "real" drone `UAV_model.slx`), and draws the
vehicle and the drone seen from the side (plane x-z). The animation is shown here and saved as a GIF for the demo / the slides.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Rectangle, Circle
from IPython.display import HTML

# Flight written by EINT_PLATFORM_eval.m -> change the path if needed
FLIGHT_CSV = r'C:\Users\ezabl\Downloads\Student_folder\Student_folder\rl_interface\platform_flight.csv'
GIF = Path('../results/figures/platform_landing.gif')

# Columns of platform_flight.csv: t z vz az_prev x vx theta wy ax az u1 u2 u3 u4 x_vehicle
F = np.loadtxt(FLIGHT_CSV, delimiter=',')
t, z, x, wy, ax, az, x_car = F[:, 0], F[:, 1], F[:, 4], F[:, 7], F[:, 8], F[:, 9], F[:, 14]
Ts, G = t[1] - t[0], 9.81

# Tilt of the drone, only for the drawing. The measured theta (column 6) is unusable (huge noise in UAV_model.slx):
# same complementary filter as platform_controller.m (gyroscope integrated + accelerometer).
theta = np.zeros_like(t)
for k in range(1, len(t)):
    theta[k] = theta[k - 1] + Ts * (wy[k] + wy[k - 1]) / 2
    if az[k] + G > 2:
        theta[k] = 0.98 * theta[k] + 0.02 * np.arctan2(ax[k], az[k] + G)
print(f'{len(t)} samples, {t[-1]:.1f} s; final gap drone - platform: {x[-1] - x_car[-1]:+.3f} m')

**Drawing** (sizes in m, chosen for readability): ground at z = −0.25 m; the vehicle is a box whose top is the platform
(z = 0, in orange), centred on the position of the vehicle; the drone is an arm of 0.3 m with two rotors, tilted by θ
(thrust along (sin θ, cos θ)).

In [ ]:
Z_GROUND, ARM = -0.25, 0.15                  # ground height, half-length of the drone arm (m)

fig, axis = plt.subplots(figsize=(10, 4))
axis.set_xlim(-0.8, max(x_car.max(), x.max()) + 1.0); axis.set_ylim(-0.4, 0.9); axis.set_aspect('equal')
axis.axhline(Z_GROUND, color='k', lw=1.5); axis.set_xlabel('x [m]'); axis.set_ylabel('z [m]'); axis.grid(alpha=0.3)

body = Rectangle((0, Z_GROUND + 0.06), 0.9, 0.19, fc='lightsteelblue', ec='k')    # vehicle body
platform = Rectangle((0, -0.02), 0.4, 0.02, fc='orange', ec='k')                    # landing platform (top = 0 m)
wheels = [Circle((0, Z_GROUND + 0.05), 0.05, fc='dimgray') for _ in range(2)]
for p in [body, platform, *wheels]:
    axis.add_patch(p)
arm, = axis.plot([], [], 'k-', lw=3)                                  # drone
rotors, = axis.plot([], [], 'o', ms=9, mfc='tab:red', mec='k')
trail, = axis.plot([], [], ':', color='tab:red', lw=1)                # path of the drone
label = axis.text(0.01, 0.95, '', transform=axis.transAxes, va='top', family='monospace')

def draw(k):
    xc = x_car[k]
    body.set_x(xc - 0.3); platform.set_x(xc - 0.2)                    # platform centred on the vehicle position
    wheels[0].center = (xc - 0.15, Z_GROUND + 0.05); wheels[1].center = (xc + 0.45, Z_GROUND + 0.05)
    c, s = np.cos(theta[k]), np.sin(theta[k])
    ends_x = [x[k] - ARM * c, x[k] + ARM * c]; ends_z = [z[k] + ARM * s, z[k] - ARM * s]   # arm perpendicular to the thrust
    arm.set_data(ends_x, ends_z); rotors.set_data(ends_x, ends_z)
    trail.set_data(x[:k + 1], z[:k + 1])
    label.set_text(f't = {t[k]:4.2f} s   drone - platform: {x[k] - x_car[k]:+.2f} m   height: {z[k]:+.2f} m')
    return [body, platform, *wheels, arm, rotors, trail, label]

anim = animation.FuncAnimation(fig, draw, frames=len(t), interval=Ts * 1000, blit=True)
GIF.parent.mkdir(parents=True, exist_ok=True)
anim.save(GIF, writer=animation.PillowWriter(fps=int(round(1 / Ts))))   # real time: 20 images per second
plt.close(fig)
print('saved:', GIF.resolve())
HTML(anim.to_jshtml())